# 05a · Transfer Learning `[kaggle]` (smoke-tests locally)

**Goal:** take an ImageNet-pretrained ResNet-50 and adapt it to new tasks four different ways, then measure which works best depending on **how much data you have** and **how different your images are from ImageNet**.

| Section | What you'll do |
|---|---|
| 🧠 Intuition | Why pretrained features transfer; the strategy ladder |
| 📐 Math | Freezing, discriminative learning rates, LoRA's low-rank update |
| 🔨 From scratch | Linear probe on cached features; LoRA for conv layers; freezing BatchNorm correctly |
| 🚀 Real run | 4 strategies × 3 data fractions × 2 datasets (Oxford Pets, EuroSAT satellite images) |
| 📊 Evaluate | Results table and plots: accuracy vs. data, trainable parameters, training time |
| 🧪 Experiment | Learning rate for full fine-tuning, domain-appropriate augmentation |

### Two run modes
- **On Kaggle (GPU):** `FULL` mode, the real study, ~30–40 min on a T4. Turn on *Internet* and *GPU T4*, and attach your `cvlib` dataset (see `00_setup`).
- **On your Mac:** `SMOKE` mode, tiny subsets and 1 epoch, ~3–4 min. It checks every code path, but **the smoke numbers mean nothing**. Your Mac's GPU fine-tunes ResNet-50 at only ~35 images/s.

In [ ]:
# ── Standard first cell ──
import sys, pathlib
for _p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents, *pathlib.Path("/kaggle/input").glob("*")]:
    if (_p / "cvlib").is_dir():
        sys.path.insert(0, str(_p)); break
from cvlib.env import setup
env = setup()

import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
from torchvision import datasets
from cvlib.datasets import cache_images, stratified_split
from cvlib.loaders import GPULoader, random_resized_crop_flip, center_resize
from cvlib.lora import add_lora, count_params
from cvlib import train as cvtrain

device = env.device
FULL = env.platform == "kaggle" or os.environ.get("CV_FULL") == "1"
CFG = dict(
    input=224 if FULL else 160,
    fractions=(0.01, 0.1, 1.0) if FULL else (0.1, 1.0),
    epochs={0.01: 25, 0.1: 12, 1.0: 6} if FULL else {0.1: 1, 1.0: 1},
    max_train=None if FULL else 400,           # cap the training pool in smoke mode
    max_test=None if FULL else 400,
    batch=64 if FULL else 32,
    amp=device == "cuda",                      # mixed precision: big win on NVIDIA, not on Apple GPUs (see 05b)
)
BACKBONE = "resnet50.a1_in1k"
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
print("MODE:", "FULL" if FULL else "SMOKE (numbers are not meaningful)", "|", CFG)

---
# 🧠 1 · Intuition

### Why transfer works
An ImageNet model's early layers detect edges, colors and textures (you saw them in modules 03–04). Those are useful for almost **any** images. Middle layers detect parts and patterns. Only the last layers are specific to ImageNet's 1,000 classes. So instead of learning from scratch, **reuse** the general layers and **re-learn** the specific ones.

### The strategy ladder (cheapest → most flexible)

| Strategy | What's trained | When it shines |
|---|---|---|
| **Linear probe** | only a new final layer, on frozen features | very little data; images similar to ImageNet |
| **Partial fine-tune** | the last stage (`layer4`) + new head | moderate data |
| **Full fine-tune** | everything, with a smaller LR for the backbone | lots of data, or images very different from ImageNet |
| **LoRA** | small low-rank "patches" added to frozen layers + head | many tasks sharing one backbone; limited memory; LLM/VLM-sized models |

### Two things decide the winner
1. **Data size:** with 1 image per class, a big model fine-tuned end-to-end overfits immediately. Frozen features are safer.
2. **Domain gap:** pet photos look like ImageNet photos (ImageNet even contains dog and cat breeds). **Satellite images** (EuroSAT) do not: top-down views, no "up", fields and rivers instead of objects. The bigger the gap, the more the backbone itself needs to adapt.

---
# 📐 2 · The math

### Freezing
A frozen parameter gets `requires_grad=False`: no gradient, no update, no optimizer memory. Frozen **BatchNorm layers must also be put in `eval()` mode**, otherwise they keep updating their running statistics from the new data even though their weights are frozen. A classic silent bug.

### Discriminative learning rates
Early layers are already good, so change them gently; the new head starts random, so train it fast:

$$\eta_{\text{backbone}} = \eta_{\text{head}} / 10 \qquad \text{(a common default)}$$

In PyTorch: one optimizer with two **parameter groups**, each with its own LR.

### LoRA: a low-rank update
For a frozen weight $W \in \mathbb{R}^{d_{out} \times d_{in}}$, learn

$$W' = W + \frac{\alpha}{r}\, B A, \qquad A \in \mathbb{R}^{r \times d_{in}},\; B \in \mathbb{R}^{d_{out} \times r},\; r \ll d_{in}, d_{out}$$

- Trainable parameters per layer: $r(d_{in} + d_{out})$ instead of $d_{in} d_{out}$. For a 512→512 layer with $r = 8$: 8k instead of 262k.
- $B$ starts at **zero**, so training starts exactly at the pretrained model.
- After training, $BA$ can be **merged** into $W$: zero extra inference cost.
- For a conv: $A$ = a $k\times k$ conv down to $r$ channels, $B$ = a 1×1 conv back up.

The bet behind LoRA: the *change* needed to adapt a model is low-rank, even if the model itself isn't. It's the standard way to fine-tune huge language and vision-language models (module 13).

### OneCycle schedule
Warm up the LR from small to max over the first ~10% of steps, then anneal smoothly to ~0. Warmup protects the pretrained weights from large, random early gradients coming back from the fresh head.

---
# 3 · Data: Oxford Pets and EuroSAT

- **Oxford-IIIT Pets:** 37 cat and dog breeds, ~7,400 photos. Close to ImageNet.
- **EuroSAT:** 27,000 Sentinel-2 satellite tiles, 64×64 px, 10 land-use classes (forest, river, highway, ...). Far from ImageNet.

Each image is decoded and resized **once** into a cached array (`cvlib.datasets.cache_images`); training then crops, flips and resizes batches directly on the GPU.

In [ ]:
t0 = time.perf_counter()
pets_tr = datasets.OxfordIIITPet(env.data_dir, split="trainval", download=True)
pets_te = datasets.OxfordIIITPet(env.data_dir, split="test", download=True)
Xp_pool, yp_pool = cache_images("pets_trainval", pets_tr, 256)
Xp_test, yp_test = cache_images("pets_test", pets_te, 256)

euro = datasets.EuroSAT(env.data_dir, download=True)
Xe_all, ye_all = cache_images("eurosat", euro, 64)
e_pool, e_test = stratified_split(ye_all, (0.6, 0.3), seed=0)           # 60% train pool, 30% test (10% unused)
print(f"cached in {time.perf_counter() - t0:.0f}s")

DATA = {
    "pets": dict(pool=(Xp_pool, yp_pool), test=(Xp_test, yp_test), classes=pets_tr.classes),
    "eurosat": dict(pool=(Xe_all[e_pool], ye_all[e_pool]), test=(Xe_all[e_test], ye_all[e_test]), classes=euro.classes),
}
for k, d in DATA.items():
    print(f"{k:8s}: {len(d['classes'])} classes | train pool {len(d['pool'][1]):>6} | test {len(d['test'][1]):>6} | stored at {d['pool'][0].shape[1]}px")

fig, axes = plt.subplots(2, 8, figsize=(13, 3.6))
rng = np.random.default_rng(1)
for row, (k, d) in enumerate(DATA.items()):
    for ax, i in zip(axes[row], rng.choice(len(d["pool"][1]), 8, replace=False)):
        ax.imshow(d["pool"][0][i]); ax.set_title(d["classes"][d["pool"][1][i]][:16], fontsize=7); ax.axis("off")
plt.suptitle("top: Pets (ImageNet-like) | bottom: EuroSAT (very un-ImageNet-like)"); plt.tight_layout(); plt.show()

In [ ]:
def take(arr_pair, idx):
    X, y = arr_pair
    return np.asarray(X[idx]), np.asarray(y[idx])

def subsample(y, frac, seed=0, cap=None):
    """A class-balanced fraction of the pool (at least 1 image per class)."""
    r = np.random.default_rng(seed)
    idx = np.concatenate([r.permutation(np.nonzero(y == c)[0])[:max(1, round(frac * (y == c).sum()))] for c in np.unique(y)])
    if cap is not None and len(idx) > cap:
        idx = r.choice(idx, cap, replace=False)
    return np.sort(idx)

def loaders_for(name, frac, train_aug=None):
    d = DATA[name]
    tr_idx = subsample(d["pool"][1], frac, cap=CFG["max_train"])
    te_idx = np.arange(len(d["test"][1]))
    if CFG["max_test"]:
        te_idx = np.random.default_rng(0).choice(te_idx, min(CFG["max_test"], len(te_idx)), replace=False)
    Xtr, ytr = take(d["pool"], tr_idx); Xte, yte = take(d["test"], te_idx)
    crop = 1.0 if name == "eurosat" else 0.875                          # satellite tiles: keep the whole tile
    train_aug = train_aug or random_resized_crop_flip(CFG["input"])
    train = GPULoader(Xtr, ytr, batch_size=CFG["batch"], shuffle=True, augment=train_aug,
                      mean=MEAN, std=STD, device=device, drop_last=len(ytr) > CFG["batch"])
    test = GPULoader(Xte, yte, batch_size=128, augment=center_resize(CFG["input"], crop), mean=MEAN, std=STD, device=device)
    return train, test, len(ytr)

---
# 🔨 4 · From scratch

### 4.1 · Linear probe on cached features

If the backbone is frozen, its output for each image never changes, so compute the features **once** and train only a logistic-regression layer on them. Seconds instead of minutes. (No augmentation this way: a fair trade for a probe.)

In [ ]:
@torch.no_grad()
def extract_features(model, loader):
    model.eval()
    feats, labels = [], []
    for x, y in loader:
        with cvtrain.autocast(device, CFG["amp"]):
            f = model.forward_head(model.forward_features(x), pre_logits=True)   # timm: pooled features before the classifier
        feats.append(f.float().cpu()); labels.append(y.cpu())
    return torch.cat(feats), torch.cat(labels)

def train_linear(Ftr, ytr, n_classes, epochs=150, lr=1e-2, wd=1e-4):
    head = nn.Linear(Ftr.shape[1], n_classes)
    opt = torch.optim.AdamW(head.parameters(), lr=lr, weight_decay=wd)
    mu, sd = Ftr.mean(0), Ftr.std(0) + 1e-6                            # standardize features
    for _ in range(epochs):                                              # full-batch: the features fit in memory
        loss = F.cross_entropy(head((Ftr - mu) / sd), ytr)
        opt.zero_grad(); loss.backward(); opt.step()
    return lambda Fx: head((Fx - mu) / sd)

backbone = timm.create_model(BACKBONE, pretrained=True, num_classes=0).to(device)   # num_classes=0 → no head
results = []
for name in DATA:
    test_feats = None
    for frac in CFG["fractions"]:
        t0 = time.perf_counter()
        tr_l, te_l, n_tr = loaders_for(name, frac)
        tr_plain = GPULoader(tr_l.x.permute(0, 2, 3, 1).cpu().numpy(), tr_l.y.cpu().numpy(), batch_size=128,
                             augment=center_resize(CFG["input"], 1.0 if name == "eurosat" else 0.875), mean=MEAN, std=STD, device=device)
        Ftr, ytr = extract_features(backbone, tr_plain)
        if test_feats is None:
            test_feats = extract_features(backbone, te_l)
        clf = train_linear(Ftr, ytr, len(DATA[name]["classes"]))
        with torch.no_grad():
            acc = (clf(test_feats[0]).argmax(1) == test_feats[1]).float().mean().item()
        results.append(dict(dataset=name, strategy="linear probe", fraction=frac, n_train=n_tr,
                            trainable=Ftr.shape[1] * len(DATA[name]["classes"]), test_acc=acc, seconds=time.perf_counter() - t0))
        print(f"{name:8s} {frac:>5.0%} ({n_tr:>5} imgs) linear probe → test acc {acc:.4f} | {time.perf_counter() - t0:.0f}s")
del backbone

### 4.2 · LoRA for convolutions

`cvlib/lora.py` contains the implementation (the code is short: read it). Let's check its two key promises on a real layer: **no change at the start**, and **far fewer trainable parameters**.

In [ ]:
import inspect
from cvlib.lora import LoRAConv2d
print(inspect.getsource(LoRAConv2d))

conv = nn.Conv2d(512, 512, 3, padding=1, bias=False)
lora_conv = LoRAConv2d(conv, r=8)
x_t = torch.randn(2, 512, 14, 14)
print(f"output identical at init: {torch.allclose(conv(x_t), lora_conv(x_t))}")
tr_p, tot_p = count_params(lora_conv)
print(f"trainable {tr_p:,} of {tot_p:,} parameters ({tr_p / tot_p:.1%})")

### 4.3 · Freezing BatchNorm properly

`model.train()` switches **every** BatchNorm back to training mode, frozen or not. This small wrapper re-freezes the BN layers whose parameters are frozen every time `.train()` is called (which `cvlib.train.fit` does each epoch).

In [ ]:
class FreezeBN(nn.Module):
    """Wrap a model so BatchNorm layers with frozen parameters stay in eval mode during training."""
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x):
        return self.model(x)

    def train(self, mode=True):
        super().train(mode)
        for m in self.model.modules():
            if isinstance(m, nn.modules.batchnorm._BatchNorm) and not any(p.requires_grad for p in m.parameters()):
                m.eval()
        return self

---
# 🚀 5 · The study: partial, full and LoRA fine-tuning

In [ ]:
def build(strategy, n_classes, lr=1e-3):
    m = timm.create_model(BACKBONE, pretrained=True, num_classes=n_classes)   # new random head (m.fc)
    head = list(m.fc.parameters())
    if strategy == "partial":
        for p in m.parameters(): p.requires_grad_(False)
        for p in [*m.layer4.parameters(), *head]: p.requires_grad_(True)
        groups = [{"params": [p for p in m.layer4.parameters()], "lr": lr / 10}, {"params": head, "lr": lr}]
    elif strategy == "full":
        backbone = [p for n, p in m.named_parameters() if not n.startswith("fc.")]
        groups = [{"params": backbone, "lr": lr / 10}, {"params": head, "lr": lr}]             # discriminative LRs
    elif strategy == "lora":
        add_lora(m, r=8, alpha=16, target=lambda n, mod: n.startswith(("layer3", "layer4")) and isinstance(mod, nn.Conv2d))
        for p in head: p.requires_grad_(True)
        lora_params = [p for n, p in m.named_parameters() if p.requires_grad and not n.startswith("fc.")]
        groups = [{"params": lora_params, "lr": lr}, {"params": head, "lr": lr}]
    return FreezeBN(m), groups

def run(name, strategy, frac, lr=1e-3, train_aug=None, verbose=False):
    tr_l, te_l, n_tr = loaders_for(name, frac, train_aug)
    model, groups = build(strategy, len(DATA[name]["classes"]), lr)
    epochs = CFG["epochs"][frac]
    opt = torch.optim.AdamW(groups, weight_decay=0.05)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=[g["lr"] for g in groups], total_steps=epochs * len(tr_l), pct_start=0.1)
    t0 = time.perf_counter()
    h = cvtrain.fit(model, tr_l, te_l, nn.CrossEntropyLoss(label_smoothing=0.1), opt, epochs=epochs, device=device,
                    batch_scheduler=sched, amp=CFG["amp"], verbose=verbose)
    trainable, _ = count_params(model)
    out = dict(dataset=name, strategy=strategy, fraction=frac, n_train=n_tr, trainable=trainable,
               test_acc=h["val_acc"][-1], seconds=time.perf_counter() - t0)
    del model; torch.cuda.empty_cache() if device == "cuda" else None
    return out

for name in DATA:
    for frac in CFG["fractions"]:
        for strategy in ("partial", "full", "lora"):
            r = run(name, strategy, frac)
            results.append(r)
            print(f"{name:8s} {frac:>5.0%} ({r['n_train']:>5} imgs) {strategy:<8} → test acc {r['test_acc']:.4f} | "
                  f"{r['trainable'] / 1e6:5.2f}M trainable | {r['seconds']:.0f}s")

df = pd.DataFrame(results)
df.to_csv(env.out_dir / f"transfer_results_{'full' if FULL else 'smoke'}.csv", index=False)

> **Note:** "test acc" here is measured on the held-out test split after a fixed recipe. We never use it to pick hyperparameters, so it stays an honest estimate. (Choosing settings would need a separate validation split, as in Experiment 1.)

---
# 📊 6 · Evaluate

In [ ]:
order = ["linear probe", "lora", "partial", "full"]
for name in DATA:
    t = df[df.dataset == name].pivot(index="strategy", columns="fraction", values="test_acc").reindex(order)
    t.columns = [f"{c:.0%} data" for c in t.columns]
    print(f"\n{name}: test accuracy")
    print(t.round(4).to_string())

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, name in zip(axes[:2], DATA):
    for s in order:
        sub = df[(df.dataset == name) & (df.strategy == s)].sort_values("fraction")
        ax.plot(sub.n_train, sub.test_acc, marker="o", label=s)
    ax.set_xscale("log"); ax.set_xlabel("training images (log scale)"); ax.set_ylabel("test accuracy")
    ax.set_title(name); ax.grid(alpha=0.3); ax.legend()
big = df[df.fraction == max(CFG["fractions"])]
for s in order:
    sub = big[big.strategy == s]
    axes[2].scatter(sub.trainable / 1e6, sub.test_acc, s=60, label=s)
axes[2].set_xscale("log"); axes[2].set_xlabel("trainable parameters (M, log)"); axes[2].set_ylabel("test accuracy")
axes[2].set_title("accuracy vs. trainable params (largest data fraction)"); axes[2].grid(alpha=0.3); axes[2].legend()
plt.tight_layout(); plt.show()

print("\ntraining time (s):")
print(df.pivot_table(index="strategy", columns=["dataset", "fraction"], values="seconds").reindex(order).round(0).to_string())

### How to read your results

Check your table against these typical patterns (the exact numbers depend on the run; the smoke-mode numbers mean nothing):

1. **Pets, little data:** the linear probe is already strong, because ImageNet's features already separate breeds. Full fine-tuning on 1% of the data (~1 image per class) tends to overfit or barely beat the probe.
2. **EuroSAT:** the linear probe lags far behind fine-tuning. Frozen ImageNet features don't describe fields and rivers well, so the backbone itself must adapt. That's the **domain gap** in one plot.
3. **More data → fine-tuning pulls ahead,** on both datasets.
4. **LoRA** usually lands between partial and full fine-tuning while training a small fraction of the parameters. Its real advantage is practical: one shared backbone with a tiny adapter per task.

**Decision guide:** start with a linear probe (minutes, a strong baseline). Then full fine-tuning with discriminative LRs if you have enough data or a domain gap. Use LoRA when memory, storage or many-tasks-one-model matter, which is exactly the situation with the big VLMs in module 13.

---
# 🧪 7 · Experiments: predict first, then run

Each one is a single extra run at the 10% data fraction.

**Experiment 1: the learning rate for full fine-tuning.** Fine-tuning with the head's LR (1e-3) on the whole backbone, vs. our 10× smaller backbone LR, vs. 10× smaller again. Predict which damages the pretrained features.

In [ ]:
# Prediction: ...
for lr in (1e-2, 1e-3, 1e-4):                                            # head LR; backbone gets lr / 10
    r = run("pets", "full", 0.1, lr=lr)
    print(f"full fine-tune, head lr {lr:.0e} (backbone {lr / 10:.0e}) → pets test acc {r['test_acc']:.4f}")
# Too high a backbone LR wrecks the pretrained features in the first steps ("catastrophic forgetting").
# Too low and nothing adapts. Proper practice: choose the LR on a validation split, never on test.

**Experiment 2: augmentation that fits the domain.** Satellite tiles have no "up": rotating by 90° or flipping vertically gives an equally valid image. Pet photos do have an up. Predict the effect of adding random 90° rotations + vertical flips on EuroSAT.

In [ ]:
# Prediction: ...
def with_rot_flip(base):
    """Add a random 90° rotation and up-down flip (one choice per batch, to keep it simple)."""
    def aug(x):
        x = torch.rot90(base(x), int(torch.randint(0, 4, (1,))), dims=(2, 3))
        return x.flip(2) if torch.rand(1) < 0.5 else x
    return aug

base_aug = random_resized_crop_flip(CFG["input"])
for label, aug in [("crop + left-right flip", base_aug), ("+ 90° rotations + up-down flip", with_rot_flip(base_aug))]:
    r = run("eurosat", "full", 0.1, train_aug=aug)
    print(f"{label:<32} → eurosat test acc {r['test_acc']:.4f}")
# Augmentation should encode the *real* invariances of your domain. Rotating pets would teach the model
# about upside-down cats it will never see. For satellites, it's free extra data.

---
# 📝 8 · Recap

**Cheat sheet**
- Pretrained early layers are general (edges, textures), late layers are task-specific: reuse, then adapt.
- Ladder: **linear probe → partial → full fine-tune**, plus **LoRA** for parameter efficiency.
- Little data and a similar domain → probe. More data or a different domain → fine-tune.
- Linear probe trick: compute frozen features once, then train only a linear layer.
- Freezing = `requires_grad_(False)` **and** BatchNorm in `eval()`.
- Discriminative LRs (backbone ≈ head / 10) + warmup (OneCycle) protect pretrained weights.
- LoRA: $W + \frac{\alpha}{r} BA$, $B$ initialized to zero, mergeable after training.
- Augment with your domain's real invariances.

**Interview-style questions**
1. You have 200 labeled X-ray images. Which transfer strategy do you try first, and why?
2. Why use a smaller learning rate for the backbone than for the new head?
3. What happens if you freeze a layer's weights but leave its BatchNorm in training mode?
4. Explain LoRA. Why initialize $B$ to zero? What's the inference overhead after merging?
5. How does the domain gap change the value of a pretrained model?
6. Why is a linear probe a useful first experiment even if you'll end up fine-tuning?
7. Give an augmentation that helps for satellite images but hurts for street photos.

**Next:** `05b_training_craft.ipynb`, the techniques that make any of these runs train better.